# Saving And Loading Examples

## Most minimal example

In [1]:
import pydocmaker as pyd
print(pyd.__version__)

2.6.13



A minimal example showing how to save and load `Doc` objects. `Doc.save` accepts a file path (string or `pathlib.Path`) or a file-like object. When no path is provided it returns the rendered content (HTML by default). `Doc.load` accepts a JSON string/bytes, a path, or a stream.


In [2]:


import pydocmaker as pyd

doc = pyd.get_example()

# save to common formats (case-insensitive suffixes are supported)
doc.save('report.html')      # writes HTML
doc.save('report.json')      # writes JSON
doc.save('report.ipynb')     # writes an ipynb (may depend on environment)

doc2 = pyd.load('report.json') # load back in (should now be same as doc)

json_str = doc.save(format='json') # save as in-memory string in given format
loaded = pyd.load(json_str) # and load back





## Detailed Saving & Loading Example

This section demonstrates several realistic saving/loading workflows you can copy into your projects. It covers:

- saving to different formats (case-insensitive suffixes),
- saving to a path without a suffix using the `ext` parameter,
- using `pathlib.Path` objects and file-like objects,
- loading from string/bytes/paths and verifying round-trip equality.

Notes:
- `.json` is the recommended format for perfect round-trip fidelity.
- `.ipynb` output depends on the notebook backend and may fail in some environments; such cases should be caught in real code.

Below is an executable example you can run in a notebook environment.

In [3]:
# Detailed, runnable saving/loading example
import pydocmaker as pyd
from pathlib import Path
from io import StringIO

# create an example document
doc = pyd.get_example()

outdir = Path('docs_output')
outdir.mkdir(exist_ok=True)


In [4]:
# 1) Save using pathlib.Path with mixed-case suffix
html_path = outdir / 'example_report.HTML'
created_html = html_path
try:
    doc.save(html_path)  # case-insensitive suffix
    print('Wrote', html_path, 'exists=', html_path.exists())
except Exception as e:
    print('Saving to HTML failed:', e)

Wrote docs_output/example_report.HTML exists= True


### Save as HTML (preferred)
Save documents as HTML for quick human-readable previews — HTML is the default format returned by `doc.save()` when no path is provided.

In [5]:
# Save to a path without suffix and explicitly set HTML as format
noext_path = outdir / 'example_report_noext'
doc.save(str(noext_path), format='HTML')  # will create example_report_noext.html
created_html2 = noext_path.with_suffix('.html')
print('Created', created_html2.exists(), created_html2)

Created True docs_output/example_report_noext.html


### Save to a path without suffix
If you prefer to provide a base name and choose the format later, pass the `format` parameter. Here we prefer `html` as the default format for readable output.

In [6]:
# In-memory HTML string: calling `doc.save()` with no path returns an HTML string
html_str = doc.save()
print('HTML length:', len(html_str))
loaded_from_html = pyd.load(html_str)
print('Loaded from HTML ok:', isinstance(loaded_from_html, pyd.Doc))

HTML length: 10834
Loaded from HTML ok: True


In [7]:
# Load back from the file we created on disk
# (created_html or created_html2 refer to files created in earlier steps)
try:
    loaded_from_file = pyd.load(str(created_html2))
    print('Loaded from file ok:', isinstance(loaded_from_file, pyd.Doc))
except NameError:
    print('created_html2 not defined yet in this notebook run; run earlier cells first')

Loaded from file ok: True


In [8]:
# JSON round-trip (recommended for perfect fidelity)
json_str = doc.dumps()
loaded_json = pyd.load(json_str)
assert doc.dumps() == loaded_json.dumps()
print('JSON roundtrip passed')

JSON roundtrip passed



Supported save/load formats for `Doc.save`/`Doc.load`:

- `.html`, `.pyd`, `.pydoc`  — HTML serialization/serialization used by pydocmaker
- `.ipynb`                    — Jupyter notebook (may require additional environment support)
- `.json`                     — Raw document JSON (recommended for round-trip fidelity)

**Note**: Saving and Loading is fundamentally different from exporting a report (e.G. to_html), since save/load allows round trip loading and saving, while exporting makes nice documents to view in other other software and not load again.  
